# Multiline Queues of Type C

In [1]:
#Preliminary packages and definitions

import itertools 
import copy
from sage.combinat.q_analogues import *
from sage.combinat.sf.sf import *
from sage.combinat.ncsf_qsym.qsym import *
from sage.rings.rational_field import *
from sage.combinat.subset import *
from sage.combinat.permutation import *
from sage.misc.latex import *
from itertools import combinations

import sage.combinat.permutation as permutation
from sage.combinat.sf.macdonald import qt_kostka
from sage.combinat.sf.ns_macdonald import E
from sage.rings.polynomial.polydict import ETuple

coeffs_ring = ZZ['q','t']

q = coeffs_ring.gens()[0]
t = coeffs_ring.gens()[1]

coeffs_field = coeffs_ring.fraction_field()
R=PolynomialRing(coeffs_field,11,'x')
xs=list(R.gens())

sym = SymmetricFunctions(coeffs_field)
h = sym.homogeneous()
m = sym.monomial()
s = sym.schur()
e = sym.elementary()
p = sym.power()
Ht = sym.macdonald().Ht()
H = sym.macdonald().H()
MP = sym.macdonald().P()
W = sym.macdonald(t=0).P()
MJ = sym.macdonald().J()
HLP = sym.hall_littlewood(t).P()
HLQ = sym.hall_littlewood(t).Q();
HLQp = sym.hall_littlewood(t).Qp();
JJ = sym.jack().J()
qsym = QuasiSymmetricFunctions(coeffs_field)
F = qsym.Fundamental()
M = qsym.Monomial()
QS = qsym.QS()
YQS = qsym.YQS()

## Object construction

In [1]:
# ALL COORDINATES START AT 1 #

# Coordinates are given in the usual way (i,j) means i to the right, j up
def balls_coordinates(balls):
    coords = []
    w = len(balls)
    l = len(balls[0])
    for i in range(w):
        for j in range(l):
            if balls[i][j] == 1:
                coords.append([i+1,j+1,"p"])
            elif balls[i][j] == -1:
                coords.append([i+1,j+1,"n"]) 
    return(coords)


# find the queue index in which elem is
def find_queue(self,elem):
    ind = -1
    for i in range(len(self.queues)):
        queue = self.queues[i]
        if elem in queue:
            ind = i
            break
    if ind == -1:
        return("element not queued")
    else:
        return(ind)  

    
# word is a ternary word with 0,1,2 and 1=(, 2=) and 0=blank, to perform parenthesis matching algorithm
# returns the leftover letters from row1 after matching all possible 1s and 2s, and deleting unmatched 2s
def match_parenthesis(w):
    word = w.copy()
    boo = True
    while boo:
        p1 = -1
        p2 = -1
        for i in range(len(word)):
            if word[i] == 1:
                p1 = i
            elif word[i] == 2:
                p2 = i
            if p2 != -1:
                word[p2] = 0
                if p1 != -1:
                    word[p1] = 0        
                break
        if p2==-1:
            boo = False
    return(word)




# Print a list of lists in a nice way
def print_list_of_lists(L):
    #print("--------------------------")
    print('\n'.join(' '.join('%2d' % x for x in l) for l in L))
    #print("--------------------------")


## TASEP methods ##

# finds the type C row from a given valid type C word
def row_from_word(w,n):
    balls = [[0 for l in range(n)] for h in range(2)]
    positions = []
    for i in range(1,n+1):
        if i in w and (-1)*i not in w:
            balls[0][i-1] = 1
            balls[1][i-1] = 1

        elif i not in w and (-1)*i in w:
            balls[0][i-1] = -1
            balls[1][i-1] = -1

        elif i in w and (-1)*i in w:
            positions.append(i)
            balls[0][i-1] = -1
            balls[1][i-1] = 1

    for j in positions:
        empty_spots_left = [k for k in range(j) if (balls[0][k] == 0) and (balls[1][k] == 0)]
        m = empty_spots_left[-1]
        balls[0][m] = 1
        balls[1][m] = -1

    return(balls)


# sorts the pieces of word so that each one is a valid type C word
def fix_pieces(word,shape):
    sh = shape
    pieces = []
    i = 0
    for s in sh:
        pieces.append(word[i:i+s])
        i += s

    ans = []
    for piece in pieces:
        fixed_piece = sorted([x for x in piece if x > 0])+sorted([x for x in piece if x < 0])
        ans += fixed_piece

    return(ans)
        

# finds the type C word from a type C row
def word_from_row(balls):
    n = len(balls[0])
    w = []
    for i in range(n):
        if balls[0][i] == 1 and balls[1][i] == 1:
            w.append(i+1)
            
        elif balls[0][i] == -1 and balls[1][i] == -1:
            w.append((-1)*(i+1))

        elif balls[0][i] == -1 and balls[1][i] == 1:
            w.append((-1)*(i+1))
            w.append(i+1)

    poss = sorted([x for x in w if x > 0])
    negs = sorted([x for x in w if x < 0])
    
    return(poss+negs)     


# checks if a word is valid as a KN column
def is_type_C_valid_word(word,n):
    ans = True
    for k in range(1,n+1):
        L = [x for x in word if abs(x) <= k]
        if len(L) > k:
            ans = False
            break
    return(ans)


# splits a word into correct sizes for the given shape
def crop_by_sizes(word, shape):
    sh = shape
    result = []
    i = 0
    for s in sh:
        result.append(word[i:i+s])
        i += s
    return(result[::-1])


# computes the ball arrangement from a word given the known shape and number of columns "n"
def ball_arrangement_from_word(word,shape,n):
    sha = shape
    dps = crop_by_sizes(word,sha)
    dec_pieces = dps
    balls = []
    for piece in dec_pieces:
        row = row_from_word(piece,n)
        balls += row
    return(balls)


# computes the lowering operator "f_i" on a word w
def F_word(w,i,shape,n):
    word = w.copy()
    if i != n and i!=0:
        aux_word = []

        for x in word:
            if x==i or x == (-1)*(i+1): 
                aux_word.append(1)
            elif x==i+1 or x==(-1)*i:
                aux_word.append(2)
            else:
                aux_word.append(0)

        pm_word = parenthesis_matching(aux_word)
        
        for y in range(len(pm_word)):
            if pm_word[y] != 0:
                aux_word[y] = 0

        inds = [j for j in range(len(aux_word)) if aux_word[j] == 1]

        if len(inds) == 0:
            return(word)
        
        else:
            pos = inds[0]
            word[pos] += 1
            return(word)

    else:
        if i==0:
            aux_word = []

            for x in word:
                if x==-1: 
                    aux_word.append(1)
                elif x==1:
                    aux_word.append(2)
                else:
                    aux_word.append(0)
    
            pm_word = parenthesis_matching(aux_word)
            
            for y in range(len(pm_word)):
                if pm_word[y] != 0:
                    aux_word[y] = 0
    
            inds = [j for j in range(len(aux_word)) if aux_word[j] == 1]
    
            if len(inds) == 0:
                return(word)
            
            else:
                pos = inds[0]
                word[pos] *= -1
                return(word)
                
        if i==n:
            aux_word = []

            for x in word:
                if x == n: 
                    aux_word.append(1)
                elif x == -n:
                    aux_word.append(2)
                else:
                    aux_word.append(0)
    
            pm_word = parenthesis_matching(aux_word)
            
            for y in range(len(pm_word)):
                if pm_word[y] != 0:
                    aux_word[y] = 0
    
            inds = [j for j in range(len(aux_word)) if aux_word[j] == 1]
    
            if len(inds) == 0:
                return(word)
            
            else:
                pos = inds[0]
                word[pos] *= -1
                return(word)



## TASEP with open (closed...) boundaries methods ##


# triggers a type C transition on a given site
# returns vec after the transition is triggered
def transition(vec,site):
    pos1 = vec[site-1]
    ans = [x for x in vec]
    if pos1==0:
        if site == len(vec):
            ans[site-1] = (-1)*pos1
            return(ans)
        else:
            pos2 = vec[site]
            if pos1 > pos2:
                ans[site-1] = pos2
                ans[site] = pos1
            return(ans)
    elif pos1 > 0:
        if site == len(vec):
            ans[site-1] = (-1)*pos1
            return(ans)
        else:
            pos2 = vec[site]
            if pos1 > pos2:
                ans[site-1] = pos2
                ans[site] = pos1
            return(ans)
    elif pos1<0:
        if site == 1:
            ans[0] = (-1)*pos1
            return(ans)
        else:
            pos2 = vec[site-2]
            if pos2 <= 0 and pos1 < pos2:
                ans[site-1] = pos2
                ans[site-2] = pos1
            return(ans)


# check if vec1 --> vec2 is a valid type C transition
def is_valid_transition_C(vec1,vec2):
    ans = False
    for site in range(len(vec1)):
        vect = transition(vec1,site+1)
        if vect == vec2:
            ans = True
            break       
    return(ans)

    


## Some methods for Type C multiline queues ##


# word is a ternary word with 0,1,2 and 1="(", 2=")" and 0=blank, to perform parenthesis matching algorithm
# returns a list with 0,1,2,... in pairs to indicate the pairing of the word
def parenthesis_matching(w):
    word = w.copy()
    boo = True
    v = [0 for p in range(len(w))]
    cont = 1
    while boo:
        p1 = -1
        p2 = -1
        for i in range(len(word)):
            if word[i] == 1:
                p1 = i
            elif word[i] == 2:
                p2 = i
            if p2 != -1:
                word[p2] = 0
                if p1 != -1:
                    word[p1] = 0    
                    v[p1] = cont
                    v[p2] = cont
                    cont += 1
                break
        if p2==-1:
            boo = False
    return(v)


# Given a rank-1 Type C MLQ, returns a word that identifies the colums (top-bottom) : o-o = -1, sq-sq = -2, empty = 0, sq-o = 1, o-sq = 2
def column_matching_word(balls):
    n = len(balls[0])
    word = []
    for i in range(n):
        col = [balls[1][i],balls[0][i]]
        if col == [1,1]:
            word.append(-1)
        elif col == [1,-1]:
            word.append(2)
        elif col == [-1,1]:
            word.append(1)
        elif col == [-1,-1]:
            word.append(-2)
        elif col == [0,0]:
            word.append(0)
    return(word)
    


## Combinatorial R in type C methods ##


# balls is a array of two elements representing a rank-1 TypeC MLQ
# balls[0] is the floor and balls[1] is the ceiling
def rem(balls,k,unm_list):
    u_list = copy.deepcopy(unm_list)
    u_list[k-1] = ""
    balls_prime = copy.deepcopy(balls)
    col = []
    col_in_k = [balls[0][k-1],balls[1][k-1]]
    
    # Information about the rank-1 MLQ
    w = column_matching_word(balls)
    pmw = parenthesis_matching(w)

    
    if col_in_k == [-1,-1]:
        # Answer
        col_ans = [-1,-1]
        
        # Find the interval in which the "k" column is contained
        the_list = [ x for x in range(1,max(pmw)+1) if ( [i for i, y in enumerate(pmw) if y == x][0] < k-1 and [i for i, y in enumerate(pmw) if y == x][1] > k-1 ) ]

        if len(the_list) == 0:
            #Remove the desired column
            balls_prime[0][k-1] = 0
            balls_prime[1][k-1] = 0
            u_list[k-1] = ""
            
        else:
            l = max( the_list )
            endpoints = [i for i, y in enumerate(pmw) if y == l] 
            
            range_of_k = range(endpoints[0],k)

            #Remove the desired column
            balls_prime[0][k-1] = 0
            balls_prime[1][k-1] = 0
    
            #Correct the o-sq (bot-top) to the left of "k" inside the range to delete the gap
    
            # Find o-sq columns
            cols_to_move = []
            for j in range_of_k:
                if balls[0][j] == 1 and balls[1][j] == -1:
                    cols_to_move.append(j)
    
            new_positions = cols_to_move[1:]+[k-1]

            for r in range(len(new_positions)):
                new_pos = new_positions[r]
                old_pos = cols_to_move[r]

                balls_prime[0][new_pos] = copy.deepcopy(balls[0][old_pos])
                balls_prime[1][new_pos] = copy.deepcopy(balls[1][old_pos])

            balls_prime[0][cols_to_move[0]] = 0
            balls_prime[1][cols_to_move[0]] = 0


            # Fix unmatched vector

            labs_to_move = []
            for j in range_of_k:
                if balls[0][j] == 1 and unm_list[j] == "u":
                    labs_to_move.append(j) 

            if len(labs_to_move) != 0:
                new_positions = labs_to_move[1:]+[k-1]

                for r in range(len(new_positions)):
                    new_pos = new_positions[r]
                    old_pos = labs_to_move[r]
                    u_list[new_pos] = unm_list[old_pos]
            
                u_list[labs_to_move[0]] = ""

    elif col_in_k == [1,1]:
        # Answer
        col_ans = [1,1]
        
        # Find the interval in which the "k" column is contained
        the_list = [ x for x in range(1,max(pmw)+1) if ( [i for i, y in enumerate(pmw) if y == x][0] < k-1 and [i for i, y in enumerate(pmw) if y == x][1] > k-1 ) ]

        if len(the_list) == 0:
            #Remove the desired column
            balls_prime[0][k-1] = 0
            balls_prime[1][k-1] = 0
            u_list[k-1] = ""
            
        else:
            l = max( the_list )
            endpoints = [i for i, y in enumerate(pmw) if y == l] 
            range_of_k = range(k-1,endpoints[1]+1)
    
            #Remove the desired column
            balls_prime[0][k-1] = 0
            balls_prime[1][k-1] = 0
    
            #Correct the sq-o (bot-top) to the right of "k" inside the range to delete the gap
    
            # Find sq-o columns
            cols_to_move = []
            for j in range_of_k:
                if balls_prime[0][j] == -1 and balls_prime[1][j] == 1:
                    cols_to_move.append(j)

            if len(cols_to_move) == 0:
                for j in range_of_k:
                    if balls_prime[0][j] == -1 and balls_prime[1][j] == 1:
                        cols_to_move.append(j)
    
            new_positions = [k-1]+cols_to_move[:-1]

            for r in range(len(new_positions)):
                new_pos = new_positions[r]
                old_pos = cols_to_move[r]

                balls_prime[0][new_pos] = copy.deepcopy(balls[0][old_pos])
                balls_prime[1][new_pos] = copy.deepcopy(balls[1][old_pos])

            balls_prime[0][cols_to_move[-1]] = 0
            balls_prime[1][cols_to_move[-1]] = 0
           
            # Fix unmatched vector

            labs_to_move = []
            for j in range_of_k:
                if balls[0][j] == -1 and unm_list[j] == "u":
                    labs_to_move.append(j) 
                
            if len(labs_to_move) != 0:
                new_positions = labs_to_move[1:]+[k-1]

                for r in range(len(new_positions)):
                    new_pos = new_positions[r]
                    old_pos = labs_to_move[r]
                    u_list[new_pos] = unm_list[old_pos]
            
                u_list[labs_to_move[-1]] = ""

    
    elif col_in_k == [-1,1]:
        # Answer
        col_ans = [-1,-1]

        # Find the complementary column to "k"
        par_index = pmw[k-1]
        k_prime = [i for i, y in enumerate(pmw) if y == par_index][0]+1
        
        # Find the interval in which the "k" column is contained
        the_list = [ x for x in range(1,max(pmw)+1) if ( [i for i, y in enumerate(pmw) if y == x][0] < k-1 and [i for i, y in enumerate(pmw) if y == x][1] > k-1 ) ]
        
        if len(the_list) == 0 and k_prime == k+1:
            #Replace the desired column
            balls_prime[0][k-1] = 1
            balls_prime[1][k-1] = 1
            balls_prime[0][k_prime-1] = 0
            balls_prime[1][k_prime-1] = 0
            u_list[k-1] = unm_list[k_prime-1]
            u_list[k_prime-1] = ""
            
        else:
            the_list = [ x for x in range(1,max(pmw)+1) if ( [i for i, y in enumerate(pmw) if y == x][0] <= k-1 and [i for i, y in enumerate(pmw) if y == x][1] >= k-1 ) ]
            l = max( the_list )
            endpoints = [i for i, y in enumerate(pmw) if y == l] 
            range_of_k = range(endpoints[0],k)
            range_of_k_prime = range(endpoints[0],k_prime)

            # print(k_prime,k)
            # print(range_of_k)
            # print(range_of_k_prime)
    
            #Delete the sqs the desired column
            balls_prime[0][k-1] = 0
            balls_prime[1][k-1] = 1
            balls_prime[0][k_prime-1] = 1
            balls_prime[1][k_prime-1] = 0
    
            #Correct the o in the bottom row to the left of "k" inside the range to delete the gap
    
            # Find o columns
            cols_to_move = []
            for j in range_of_k:
                if balls_prime[0][j] == 1:
                    cols_to_move.append(j)            
    
            new_positions = cols_to_move[1:]+[k-1]

            for r in range(len(new_positions)):
                new_pos = new_positions[r]
                old_pos = cols_to_move[r]
                balls_prime[0][new_pos] = copy.deepcopy(balls[0][old_pos])
                
            balls_prime[0][cols_to_move[0]] = 0

            # Fix unmatched vector

            labs_to_move = []
            for j in range_of_k:
                if balls[0][j] == 1 and unm_list[j] == "u":
                    labs_to_move.append(j) 

            if len(labs_to_move)==0:
                u_list[k-1] = unm_list[k_prime-1]

            else:
                new_positions = labs_to_move[1:]+[k-1]

                for r in range(len(new_positions)):
                    new_pos = new_positions[r]
                    old_pos = labs_to_move[r]
                    u_list[new_pos] = unm_list[old_pos]
            
                u_list[labs_to_move[0]] = ""
            

            #Correct the sq in the top row to the left of " k_prime " inside the range to delete the gap
    
            # Find sq columns
            cols_to_move = []
            for j in range_of_k_prime:
                if balls[1][j] == -1:
                    cols_to_move.append(j)
    
            new_positions = cols_to_move[1:]+[k_prime-1]

            for r in range(len(new_positions)):
                new_pos = new_positions[r]
                old_pos = cols_to_move[r]

                balls_prime[1][new_pos] = copy.deepcopy(balls[1][old_pos])

            balls_prime[1][cols_to_move[0]] = 0 

            

    elif col_in_k == [1,-1]:
        # Answer
        col_ans = [1,1]

        # Find the complementary column to "k"
        par_index = pmw[k-1]
        k_prime = [i for i, y in enumerate(pmw) if y == par_index][1]+1        
        
        # Find the interval in which the "k" column is contained
        the_list = [ x for x in range(1,max(pmw)+1) if ( [i for i, y in enumerate(pmw) if y == x][0] < k-1 and [i for i, y in enumerate(pmw) if y == x][1] > k-1 ) ]

        if len(the_list) == 0 and k_prime == k+1:
            #Replace the desired column
            balls_prime[0][k-1] = -1
            balls_prime[1][k-1] = -1
            balls_prime[0][k_prime-1] = 0
            balls_prime[1][k_prime-1] = 0
            u_list[k-1] = unm_list[k_prime-1]
            u_list[k_prime-1] = ""
            
        else:
            the_list = [ x for x in range(1,max(pmw)+1) if ( [i for i, y in enumerate(pmw) if y == x][0] <= k-1 and [i for i, y in enumerate(pmw) if y == x][1] >= k-1 ) ]

            l = max( the_list )
            endpoints = [i for i, y in enumerate(pmw) if y == l] 
            range_of_k = range(k-1,endpoints[1]+1)
            range_of_k_prime = range(k_prime-1,endpoints[1]+1)

            # print(k_prime,k)
            # print(range_of_k)
            # print(range_of_k_prime)
    
            #Delete the o the desired column
            balls_prime[0][k-1] = 0
            balls_prime[1][k-1] = -1
            balls_prime[0][k_prime-1] = -1
            balls_prime[1][k_prime-1] = 0
    
            #Correct the sq in the bottom row to the right of "k" inside the range to delete the gap
    
            # Find o columns
            cols_to_move = []
            for j in range_of_k:
                if balls_prime[0][j] == -1:
                    cols_to_move.append(j)            
    
            new_positions = [k-1]+cols_to_move[:-1]

            for r in range(len(new_positions)):
                new_pos = new_positions[r]
                old_pos = cols_to_move[r]
                balls_prime[0][new_pos] = copy.deepcopy(balls[0][old_pos])
                
            balls_prime[0][cols_to_move[-1]] = 0

            # Fix unmatched vector

            labs_to_move = []
            for j in range_of_k:
                if balls[0][j] == -1 and unm_list[j] == "u":
                    labs_to_move.append(j) 
            
            if len(labs_to_move)==0:
                u_list[k-1] = unm_list[k_prime-1]

            else:
                new_positions = labs_to_move[1:]+[k-1]

                for r in range(len(new_positions)):
                    new_pos = new_positions[r]
                    old_pos = labs_to_move[r]
                    u_list[new_pos] = unm_list[old_pos]
            
                u_list[labs_to_move[-1]] = ""
            

            #Correct the o in the top row to the right of " k_prime " inside the range to delete the gap
    
            # Find o columns
            cols_to_move = []
            for j in range_of_k_prime:
                if balls[1][j] == 1:
                    cols_to_move.append(j)
    
            new_positions = [k_prime-1]+cols_to_move[:-1]

            for r in range(len(new_positions)):
                new_pos = new_positions[r]
                old_pos = cols_to_move[r]

                balls_prime[1][new_pos] = copy.deepcopy(balls[1][old_pos])

            balls_prime[1][cols_to_move[-1]] = 0 

    return(balls_prime,col_ans,u_list)


#order is a list that says which particle to remove from left to right in step "x"
#order has size = |# "u"s in unm_list| = U  and 1 <= order[x] < U-x-1 (?)

def ordered_removal(balls,unm_list,order):
    U = unm_list.count("u")
    local_balls = copy.deepcopy(balls)
    local_unm_list = copy.deepcopy(unm_list)
    col_rem = []
    # print(local_balls,local_unm_list)
    for x in range(len(order)):
        pos = order[x]
        ind_u = [i for i, y in enumerate(local_unm_list) if y == "u"][pos-1]
        col_to_rem = ind_u+1
        [local_balls,col_rem,local_unm_list] = rem(local_balls,col_to_rem,local_unm_list)
        # print(local_balls,local_unm_list)
    return(local_balls)    



## New versions of the methods for Combinatorial R in type C ##


# rotates the type C row 180 degrees
def flip(balls):
    return([list(reversed(balls[1])),list(reversed(balls[0]))])



## Top and Bottom Removals ##


# attempts to remove a the particle from column "k" from the bottom (it may not be possible)
def botRem(balls,k):
    balls_prime = copy.deepcopy(balls)
    col = []
    col_in_k = [balls[0][k-1],balls[1][k-1]]
    
    # Information about the rank-1 MLQ
    w = column_matching_word(balls)
    pmw = parenthesis_matching(w)

    
    if col_in_k == [-1,-1]:
        # Answer
        col_ans = [-1,-1]
        
        # Find the interval in which the "k" column is contained
        the_list = [ x for x in range(1,max(pmw)+1) if ( [i for i, y in enumerate(pmw) if y == x][0] < k-1 and [i for i, y in enumerate(pmw) if y == x][1] > k-1 ) ]

        if len(the_list) == 0:
            #Remove the desired column
            balls_prime[0][k-1] = 0
            balls_prime[1][k-1] = 0
            
        else:
            l = max( the_list )
            endpoints = [i for i, y in enumerate(pmw) if y == l] 
            
            range_of_k = range(endpoints[0],k)

            #Remove the desired column
            balls_prime[0][k-1] = 0
            balls_prime[1][k-1] = 0
    
            #Correct the o-sq (bot-top) to the left of "k" inside the range to delete the gap
    
            # Find o-sq columns
            cols_to_move = []
            for j in range_of_k:
                if balls[0][j] == 1 and balls[1][j] == -1:
                    cols_to_move.append(j)
    
            new_positions = cols_to_move[1:]+[k-1]

            for r in range(len(new_positions)):
                new_pos = new_positions[r]
                old_pos = cols_to_move[r]

                balls_prime[0][new_pos] = copy.deepcopy(balls[0][old_pos])
                balls_prime[1][new_pos] = copy.deepcopy(balls[1][old_pos])

            balls_prime[0][cols_to_move[0]] = 0
            balls_prime[1][cols_to_move[0]] = 0

    elif col_in_k == [1,1]:
        # Answer
        col_ans = [1,1]
        
        # Find the interval in which the "k" column is contained
        the_list = [ x for x in range(1,max(pmw)+1) if ( [i for i, y in enumerate(pmw) if y == x][0] < k-1 and [i for i, y in enumerate(pmw) if y == x][1] > k-1 ) ]

        if len(the_list) == 0:
            #Remove the desired column
            balls_prime[0][k-1] = 0
            balls_prime[1][k-1] = 0
            
        else:
            l = max( the_list )
            endpoints = [i for i, y in enumerate(pmw) if y == l] 
            range_of_k = range(k-1,endpoints[1]+1)
    
            #Remove the desired column
            balls_prime[0][k-1] = 0
            balls_prime[1][k-1] = 0
    
            #Correct the sq-o (bot-top) to the right of "k" inside the range to delete the gap
    
            # Find sq-o columns
            cols_to_move = []
            for j in range_of_k:
                if balls_prime[0][j] == -1 and balls_prime[1][j] == 1:
                    cols_to_move.append(j)

            if len(cols_to_move) == 0:
                for j in range_of_k:
                    if balls_prime[0][j] == -1 and balls_prime[1][j] == 1:
                        cols_to_move.append(j)
    
            new_positions = [k-1]+cols_to_move[:-1]

            for r in range(len(new_positions)):
                new_pos = new_positions[r]
                old_pos = cols_to_move[r]

                balls_prime[0][new_pos] = copy.deepcopy(balls[0][old_pos])
                balls_prime[1][new_pos] = copy.deepcopy(balls[1][old_pos])

            balls_prime[0][cols_to_move[-1]] = 0
            balls_prime[1][cols_to_move[-1]] = 0
           
    
    elif col_in_k == [-1,1]:
        # Answer
        col_ans = [-1,-1]

        # Find the complementary column to "k"
        par_index = pmw[k-1]
        k_prime = [i for i, y in enumerate(pmw) if y == par_index][0]+1
        
        # Find the interval in which the "k" column is contained
        the_list = [ x for x in range(1,max(pmw)+1) if ( [i for i, y in enumerate(pmw) if y == x][0] < k-1 and [i for i, y in enumerate(pmw) if y == x][1] > k-1 ) ]
        
        if len(the_list) == 0 and k_prime == k+1:
            #Replace the desired column
            balls_prime[0][k-1] = 1
            balls_prime[1][k-1] = 1
            balls_prime[0][k_prime-1] = 0
            balls_prime[1][k_prime-1] = 0
            
        else:
            the_list = [ x for x in range(1,max(pmw)+1) if ( [i for i, y in enumerate(pmw) if y == x][0] <= k-1 and [i for i, y in enumerate(pmw) if y == x][1] >= k-1 ) ]
            l = max( the_list )
            endpoints = [i for i, y in enumerate(pmw) if y == l] 
            range_of_k = range(endpoints[0],k)
            range_of_k_prime = range(endpoints[0],k_prime)
            
            #Delete the sqs the desired column
            balls_prime[0][k-1] = 0
            balls_prime[1][k-1] = 1
            balls_prime[0][k_prime-1] = 1
            balls_prime[1][k_prime-1] = 0
    
            #Correct the o in the bottom row to the left of "k" inside the range to delete the gap
    
            # Find o columns
            cols_to_move = []
            for j in range_of_k:
                if balls_prime[0][j] == 1:
                    cols_to_move.append(j)            
    
            new_positions = cols_to_move[1:]+[k-1]

            for r in range(len(new_positions)):
                new_pos = new_positions[r]
                old_pos = cols_to_move[r]
                balls_prime[0][new_pos] = copy.deepcopy(balls[0][old_pos])
                
            balls_prime[0][cols_to_move[0]] = 0

            #Correct the sq in the top row to the left of " k_prime " inside the range to delete the gap
    
            # Find sq columns
            cols_to_move = []
            for j in range_of_k_prime:
                if balls[1][j] == -1:
                    cols_to_move.append(j)
    
            new_positions = cols_to_move[1:]+[k_prime-1]

            for r in range(len(new_positions)):
                new_pos = new_positions[r]
                old_pos = cols_to_move[r]

                balls_prime[1][new_pos] = copy.deepcopy(balls[1][old_pos])

            balls_prime[1][cols_to_move[0]] = 0 
            

    elif col_in_k == [1,-1]:
        # Answer
        col_ans = [1,1]

        # Find the complementary column to "k"
        par_index = pmw[k-1]
        k_prime = [i for i, y in enumerate(pmw) if y == par_index][1]+1        
        
        # Find the interval in which the "k" column is contained
        the_list = [ x for x in range(1,max(pmw)+1) if ( [i for i, y in enumerate(pmw) if y == x][0] < k-1 and [i for i, y in enumerate(pmw) if y == x][1] > k-1 ) ]

        if len(the_list) == 0 and k_prime == k+1:
            #Replace the desired column
            balls_prime[0][k-1] = -1
            balls_prime[1][k-1] = -1
            balls_prime[0][k_prime-1] = 0
            balls_prime[1][k_prime-1] = 0
            
        else:
            the_list = [ x for x in range(1,max(pmw)+1) if ( [i for i, y in enumerate(pmw) if y == x][0] <= k-1 and [i for i, y in enumerate(pmw) if y == x][1] >= k-1 ) ]

            l = max( the_list )
            endpoints = [i for i, y in enumerate(pmw) if y == l] 
            range_of_k = range(k-1,endpoints[1]+1)
            range_of_k_prime = range(k_prime-1,endpoints[1]+1)
    
            #Delete the o the desired column
            balls_prime[0][k-1] = 0
            balls_prime[1][k-1] = -1
            balls_prime[0][k_prime-1] = -1
            balls_prime[1][k_prime-1] = 0
    
            #Correct the sq in the bottom row to the right of "k" inside the range to delete the gap
    
            # Find o columns
            cols_to_move = []
            for j in range_of_k:
                if balls_prime[0][j] == -1:
                    cols_to_move.append(j)            
    
            new_positions = [k-1]+cols_to_move[:-1]

            for r in range(len(new_positions)):
                new_pos = new_positions[r]
                old_pos = cols_to_move[r]
                balls_prime[0][new_pos] = copy.deepcopy(balls[0][old_pos])
                
            balls_prime[0][cols_to_move[-1]] = 0

            #Correct the o in the top row to the right of " k_prime " inside the range to delete the gap
    
            # Find o columns
            cols_to_move = []
            for j in range_of_k_prime:
                if balls[1][j] == 1:
                    cols_to_move.append(j)
    
            new_positions = [k_prime-1]+cols_to_move[:-1]

            for r in range(len(new_positions)):
                new_pos = new_positions[r]
                old_pos = cols_to_move[r]

                balls_prime[1][new_pos] = copy.deepcopy(balls[1][old_pos])

            balls_prime[1][cols_to_move[-1]] = 0 

    return(balls_prime)


# attempts to remove a the particle from column "k" from the top (it may not be possible)
def topRem(balls,k):
    n = len(balls[0])
    return(flip(botRem(flip(balls),n-k+1)))



## Top and Bottom Insertions ##


# attempts to insert a particle of given type "typ" into column "k" from the bottom (it may not be possible)
def botIns(balls,typ,k):
    balls_prime = copy.deepcopy(balls)
    n = len(balls[0])
    its_valid = True

    if balls[0][k-1] == typ:
        its_valid = False
    else:
        if typ == 1 and len([x for x in range(k-1,n) if balls[0][x] == 0]) == 0:
            its_valid = False
            
        elif typ == -1 and len([x for x in range(k) if balls[0][x] == 0]) == 0:
            its_valid = False
    
    if not its_valid:
        return("Insertion is undefined")

    else:

        # Information about the rank-1 MLQ
        w = column_matching_word(balls)
        pmw = parenthesis_matching(w)
        
        
        if balls[0][k-1]==0:
            balls_prime[0][k-1] = typ
            balls_prime[1][k-1] = typ
            return(balls_prime)
            
        else:
            if typ == 1:
                rr = min([x for x in range(k-1,n) if balls[0][x] == 0])+1
                ll = k

                balls_prime[0][ll-1] = typ
                
                balls_prime[1][rr-1] = typ
                balls_prime[0][rr-1] = (-1)*typ
                
                return(balls_prime)

            if typ == -1:
                rr = k
                ll = max([x for x in range(k) if balls[0][x] == 0])+1

                balls_prime[0][rr-1] = typ
                
                balls_prime[1][ll-1] = typ
                balls_prime[0][ll-1] = (-1)*typ
                
                return(balls_prime)


# attempts to insert a particle of given type "typ" into column "k" from the top (it may not be possible)
def topIns(balls,typ,k):
    n = len(balls[0])
    if type(botIns(flip(balls),typ,n-k+1)) == str:
        return("Insertion is undefined")

    else:
        return(flip(botIns(flip(balls),typ,n-k+1)))



## Type C collapsing ##


# finds the pairing of two rows of circles and squares so that it does not bounce on the left
# returns the unmatched indices 
def unpaired_classical_pairing(Bpar,Tpar):
    B = copy.deepcopy(Bpar)
    T = copy.deepcopy(Tpar)
    n = len(B)

    # version 1: using MLQs: might be wrong
    
    # # find vertical pairings
    # for i in range(len(B)):
    #     if (B[i] == 1 and T[i]== 1) or (B[i] == -1 and T[i]== -1):
    #         B[i] = 0
    #         T[i] = 0
            
    # lT = len([i for i in range(len(T)) if T[i] != 0])
    # lB = len([i for i in range(len(B)) if B[i] != 0])
    
    # if lB >= lT:
    #     unmatched_indices = []
    #     M_prime =  MultilineQueueC([B,T])
    #     M_prime.pair(0)
    #     the_queues = M_prime.queues

    #     for q in [queue for queue in the_queues if len(queue) > 1]:
    #         above = q[0]
    #         below = q[1]
    #         if above[2] == 'p':
    #             if below[2] == 'n' or (below[2] == 'p' and above[1] < below[1]):
    #                 unmatched_indices.append(above[1])
    #         else:
    #             if (below[2] == 'n' and above[1] > below[1]):
    #                 unmatched_indices.append(above[1])

    #     return(unmatched_indices)

    # else:
    #     B_prime = list(reversed(T))
    #     T_prime = list(reversed(B))

    #     flip_list = []

    #     M_prime =  MultilineQueueC([B_prime,T_prime])
    #     M_prime.pair(0)
    #     the_queues = M_prime.queues

    #     for q in [queue for queue in the_queues if len(queue) > 1]:
    #         above = q[0]
    #         below = q[1]
    #         if below[2] == 'p':
    #             if above[2] == 'n' or (above[2] == 'p' and above[1] < below[1]):
    #                 flip_list.append(below[1])
    #         else:
    #             if (above[2] == 'n' and above[1] > below[1]):
    #                 flip_list.append(below[1])

    #     for q in [queue for queue in the_queues if len(queue) == 1]:
    #         flip_list.append(q[0][1])

    #     return([n-j+1 for j in flip_list])


    # version 2: using unfolding and classical parenthesis matching

    lT = len([i for i in range(len(T)) if T[i] != 0])
    lB = len([i for i in range(len(B)) if B[i] != 0])
    
    if lB >= lT:
        unf_T = [0 for x in range(2*n)]
        unf_B = [0 for y in range(2*n)]
        for i in range(n):
            if T[i] == 1:
                unf_T[i] = 1
            elif T[i] == -1: 
                unf_T[n-1+(n-i)] = 1
            
            if B[i] == 1:
                unf_B[i] = 1
            elif B[i] == -1: 
                unf_B[n-1+(n-i)] = 1
    
        # contruct parenthesis word
        par_w = []
        for j in reversed(range(2*n)):
            if unf_T[j] == 1:
                par_w.append(1)
            else:
                par_w.append(0)
    
            if unf_B[j] == 1:
                par_w.append(2)
            else:
                par_w.append(0)
    
        unm_par = list(reversed(match_parenthesis(par_w)))
        ans = []

        for k in range(len(unm_par)):
            if unm_par[k] == 1:
                if k < 2*n:
                    ans.append(int((k+1)/2.0))
                else:
                    ans.append(2*n-int((k-1)/2.0))
    
        return(ans)

    else:
        B_prime = B
        T_prime = T

        unf_T = [0 for x in range(2*n)]
        unf_B = [0 for y in range(2*n)]
        for i in range(n):
            if T_prime[i] == 1:
                unf_T[i] = 1
            elif T_prime[i] == -1: 
                unf_T[n-1+(n-i)] = 1
            
            if B_prime[i] == 1:
                unf_B[i] = 1
            elif B_prime[i] == -1: 
                unf_B[n-1+(n-i)] = 1
    
        # contruct parenthesis word
        par_w = []
        for j in reversed(range(2*n)):
            if unf_T[j] == 1:
                par_w.append(1)
            else:
                par_w.append(0)
    
            if unf_B[j] == 1:
                par_w.append(2)
            else:
                par_w.append(0)
    
        unm_par = match_parenthesis(par_w)

        ans = []
        for k in range(len(unm_par)):
            if unm_par[k] == 1:
                if k < 2*n:
                    ans.append(int((k+2)/2.0))
                else:
                    ans.append(int((4*n-k)/2.0))
    
        return(ans)
        


# looks for the farthest "to the right" unpaired particle on a row of circles and squares with a choice of unpaired particles
def farthest_unpaired(A,unm_list):
    index = 0
    search = True

    for x in unm_list:
        if A[x-1] == -1:
            index = x
            search = False
            break

    if search:
        for x in unm_list:
            if A[x-1] == 1:
                index = x
    
    return(index)
    

# applies either dropping or anhilation to a pair of type C rows 
def row_operator(X,Y):
    U0 = sorted(unpaired_classical_pairing(X[1],Y[0]))
    if len(U0) > 0:
        far_ind = farthest_unpaired(Y[0],U0)
        typ = Y[0][far_ind-1]
        balls_bot = topIns(X,typ,far_ind)
        if type(balls_bot) == str:
            return([topRem(X,far_ind),botRem(Y,far_ind)])
        else:
            return([balls_bot,botRem(Y,far_ind)])
    else:
        return([X,Y])


# applies the row operator to a pair of type C rows "m" times
def row_operator_power(X,Y,m):
    [X_prime,Y_prime] = [copy.deepcopy(X),copy.deepcopy(Y)]

    U0 = unpaired_classical_pairing(X_prime[1],Y_prime[0])

    for am in range(m):
        [X_prime,Y_prime] = row_operator(X_prime,Y_prime)
        U0 = unpaired_classical_pairing(X_prime[1],Y_prime[0])

    return([X_prime,Y_prime])


# applies the row operator to a pair of type C rows as many times as possible
def row_operator_star(X,Y):
    [X_prime,Y_prime] = [copy.deepcopy(X),copy.deepcopy(Y)]

    U0 = unpaired_classical_pairing(X_prime[1],Y_prime[0])

    while len(U0) > 0:
        [X_prime,Y_prime] = row_operator(X_prime,Y_prime)
        U0 = unpaired_classical_pairing(X_prime[1],Y_prime[0])

    return([X_prime,Y_prime])



##   MULTILINE QUEUE CLASS   ##


class MultilineQueueC:
    queues = []
    case = 0
    r = 0
    set_case = False
    
    # balls may be valid (partition content)
    # size = [width,height]
    def __init__(self,balls):
        self.balls = copy.deepcopy(balls)
        self.size = [len(balls[0]),len(balls)]
        self.balls_coordinates = balls_coordinates(balls)

        row_content = []
        for i in range(int(len(balls)/2.0)):
            row_content.append(sum([abs(x) for x in self.balls[2*i]]))
        self.shape = row_content[::-1]
        self.shape = list(reversed(self.shape))
        self.already_collapsed = False
                
        self.QM =[ [] for _ in range(int(self.size[1]/2.0))]
        
    # This method creates the queues
    # 
    # case=0:   pairing to the right with (possibly) wrapping conditions
    #           pairing is done left to right in each row and priority order is respected
    #           queues are given as sequences of coordinates starting from the top

    
    def pair(self,case):
        if not self.set_case:
            self.case = case
            self.set_case = True
        self.queues = []
        if case==0:       
            if self.partition() != []:
                to_queue = []
                w = self.size[1]
                l = self.size[0]
                for i in range(w):
                    for j in reversed(range(l)):
                        if self.balls[i][j] == 1: 
                            to_queue.append([i+1,j+1,"p"])
                        elif self.balls[i][j] == -1:
                            to_queue.append([i+1,j+1,"n"])
                cont = True
                while cont:
                    b = to_queue[-1]
                    queue = [b]
                    to_queue.remove(b)
                    done = False
                    b0 = b.copy()
                    while not done:
                        j = b0[0]-1
                        
                        #If the particle is positive, it pairs weakly to the left first
                        if b0[2] == "p":
                            l = [m for m in range(0,b0[1]+1) if ([j,m,"p"] in to_queue)]
                        #If the particle is negative, it pairs weakly to the left first
                        if b0[2] == "n":
                            l = [m for m in range(b0[1],self.size[0]+1) if ([j,m,"n"] in to_queue)]
                        
                        #if b0 is at the bottom
                        if b0[0] == 1:
                            done = True
                        #otherwise
                        else:
                            ind = ""
                            #case 1: bouncing
                            if l==[]:
                                if b0[2] == "p":
                                    negs = [m for m in range(self.size[0]+1) if ([j,m,"n"] in to_queue)]
                                    poss = [m for m in range(self.size[0]+1) if ([j,m,"p"] in to_queue)]
                                    if len(negs) > 0:
                                        c = min(negs)
                                        ind = "n"
                                    else:
                                        c = max(poss)
                                        ind = "p"
                                    
                                elif b0[2] == "n":
                                    negs = [m for m in range(self.size[0]+1) if ([j,m,"n"] in to_queue)]
                                    poss = [m for m in range(self.size[0]+1) if ([j,m,"p"] in to_queue)]
                                    if len(poss) > 0:
                                        c = max(poss)
                                        ind = "p"
                                    else:
                                        c = min(negs)
                                        ind = "n"
                            #case 2: non-wrapping
                            else:
                                if b0[2] == "p":
                                    c = max(l)
                                    ind = "p"
                                elif b0[2] == "n":
                                    c = min(l)
                                    ind = "n"
                            queue.append([j,c,ind])
                            to_queue.remove([j,c,ind])
                            b0 = [j,c,ind]
                    self.queues.append(queue)
                    if len(to_queue) == 0:
                        cont = False  

        
    # Collapsing procedure for the ball arrangement
    # this method returns the collapsed ball arrangement and updates self.QM to be the recording object
    
    def collapse_and_record(self):
        if not self.already_collapsed:
            self.already_collapsed = True
            balls_prime = copy.deepcopy(self.balls)
            L = int(self.size[1]/2.0)
            self.QM[0] += [1 for r in range(self.shape[0])]
            
            for i in range(1,L):
    
                # print(f"i={i}")
    
                ### [# of annihilations, # of droppings]
                indices = [[0,0] for r in range(L)]
    
                initShape = [len([x for x in balls_prime[2*r] if x!=0]) for r in range(i+1)]
    
                # print(f"Initial shape: {initShape}")
                
                for j in reversed(range(1,i+1)):
    
                    row1 = [balls_prime[2*j-2].copy(),balls_prime[2*j-1].copy()]
                    row2 = [balls_prime[2*j].copy(),balls_prime[2*j+1].copy()]
    
    
                    # print(row1)
                    # print(row2)
                    
                    [R1,R2] = row_operator_star(row1,row2)
    
                    totalAboveInit = len([x for x in balls_prime[2*j] if x!=0])
    
                    totalInit = len([x for x in balls_prime[2*j-2] if x!=0]) + len([x for x in balls_prime[2*j] if x!=0])
                    totalFin = len([x for x in R1[0] if x!=0]) + len([x for x in R2[0] if x!=0])
    
                    balls_prime[2*j-2] = R1[0]
                    balls_prime[2*j-1] = R1[1]
                    balls_prime[2*j] = R2[0]
                    balls_prime[2*j+1] = R2[1]
    
                    indices[j-1] = [int((totalInit-totalFin)/2.0), 
                                    totalAboveInit-int((totalInit-totalFin)/2.0)-len([x for x in R2[0] if x!=0])]
    
    
                # print(indices)
                
                finalShape = [len([x for x in balls_prime[2*r] if x!=0]) for r in range(i+1)]
    
                # print(f"Final shape: {finalShape}")
                
                for j in range(1,i+1):
                    if j==1:
                        self.QM[j-1] += [i+1 for k in range(indices[j-1][1])]
                    if j<i:
                        self.QM[j] += [i+1 for k in range(-sum(indices[j-1]) + indices[j][1])] + [-1*(i+1) for k in range(indices[j-1][0])]
                    if j==i:
                        self.QM[j] += [i+1 for k in range(finalShape[j])] + [-1*(i+1) for k in range(indices[j-1][0])]
                    
            return(balls_prime)

        else:
            return("Already collapsed!")
    

    ## PROPERTIES OF THE MULTILINE QUEUE ##


    # determines if the multiline queue has a queue that bounces on the left    
    def is_left_bouncing(self):
        ans = False
        for queue in self.queues:
            for i in range(len(queue)-1):
                if queue[i][2] == 'p':
                    if queue[i+1][2] == 'n':
                        ans = True
                        break
                    elif queue[i+1][2] == 'p' and queue[i][1] < queue[i+1][1]:
                        ans = True
                        break
                else:
                    if queue[i+1][2] == 'n' and queue[i][1] > queue[i+1][1]:
                        ans = True
                        break
            if ans:
                break
        
        return(ans)

    
    # finds the shape of the multiline queue
    def partition(self):
        lam_prime = [sum([abs(x) for x in self.balls[2*i]]) for i in range(int(self.size[1]/2.0))]
        lam_p = Partition(lam_prime)
        return(lam_p.conjugate())


    # returns the row word of the multiline queue
    def rw(self):
        w = []
        for i in reversed(range(int(len(self.balls)/2.0))):
            row = [self.balls[2*i],self.balls[2*i+1]]
            w += word_from_row(row)
        return(w)

    #returns the column word of the multiline queue: row number of the balls from top to bottom and left to right
    def cw(self):
        w = []
        for i in range(self.size[0]):
            for j in reversed(range(self.size[1])):
                if self.balls[j][i] == 1:
                    w.append(j+1)
        
        return(w)
    
    #returns the inverted column word of the multiline queue: row number of the balls from top to bottom and right to left
    def invcolw(self):
        w = []
        for i in reversed(range(self.size[0])):
            for j in reversed(range(self.size[1])):
                if self.balls[j][i] == 1:
                    w.append(j+1)
                return(w)
    
    
    ## VISUALIZATION METHODS ##
        
    
    # Draw a queue between pi and pf in a drawing win
    # queues must be ordered from top to bottom
    # case : see pairing cases
    def draw_queue(self,win,pi,pf,case,offset):
        r = 0.2
        xi = pi[1]-0.5
        yi = pi[0]-0.5
        xf = pf[1]-0.5
        yf = pf[0]-0.5
        ti = pi[2]
        tf = pf[2]

        if ti == tf:
            if case == 0:
                if ti == 'p':
                    if xi < xf:
                        win += line([(xi,yi-self.r),(xi,yi-self.r-offset)],color=Color('purple'),linestyle='-.')
                        win += line([(xi,yi-self.r-offset),(xf,yf+self.r+offset)],color=Color('purple'),linestyle='-.')
                        win += line([(xf,yf+self.r+offset),(xf,yf+self.r)],color=Color('purple'),linestyle='-.')
                    else:
                        win += line([(xi,yi-self.r),(xi,yi-self.r-offset)],color=Color('black'))
                        win += line([(xi,yi-self.r-offset),(xf,yf+self.r+offset)],color=Color('black'))
                        win += line([(xf,yf+self.r+offset),(xf,yf+self.r)],color=Color('black'))
                    return(win)

                if ti == 'n':
                    if xi > xf:
                        win += line([(xi,yi-self.r),(xi,yi-self.r-offset)],color=Color('purple'),linestyle='-.')
                        win += line([(xi,yi-self.r-offset),(xf,yf+self.r+offset)],color=Color('purple'),linestyle='-.')
                        win += line([(xf,yf+self.r+offset),(xf,yf+self.r)],color=Color('purple'),linestyle='-.')
                    else:
                        win += line([(xi,yi-self.r),(xi,yi-self.r-offset)],color=Color('black'))
                        win += line([(xi,yi-self.r-offset),(xf,yf+self.r+offset)],color=Color('black'))
                        win += line([(xf,yf+self.r+offset),(xf,yf+self.r)],color=Color('black'))
                    return(win)

        if ti == 'p' and tf == 'n':
            if case == 0:
                win += line([(xi,yi-self.r),(xi,yi-self.r-offset)],color=Color('red'),linestyle='--')
                win += line([(xi,yi-self.r-offset),(xf,yf+self.r+offset)],color=Color('red'),linestyle='--')
                win += line([(xf,yf+self.r+offset),(xf,yf+self.r)],color=Color('red'),linestyle='--')
            return(win)

        if ti == 'n' and tf == 'p':
            if case == 0:
                win += line([(xi,yi-self.r),(xi,yi-self.r-offset)],color=Color('blue'),linestyle=':')
                win += line([(xi,yi-self.r-offset),(xf,yf+self.r+offset)],color=Color('blue'),linestyle=':')
                win += line([(xf,yf+self.r+offset),(xf,yf+self.r)],color=Color('blue'),linestyle=':')
            return(win)

            
    # Gives the drawing of a MLQ that can be paired or not (in which case just shows the ball arrangement)             
    def draw(self,radius):
        win = Graphics()
        self.r = radius
        w = self.size[1]
        h = self.size[0]
        
        #Draw gray grid        
        for i in range(0,h+1,1):
            win += line([(i,0),(i,w)],color=Color('lightgray'))
        for i in range(0,w+1,1):    
            win += line([(0,i),(h,i)],color=Color('lightgray'))
        
        #Draw ball arrangement
        for b in self.balls_coordinates:
            yb = b[0]
            xb = b[1]
            stop = False
            lab = str("")
            for queue in self.queues:
                if(b in queue):
                    lab = str(len(queue))
                if(stop):
                    break
                
            if len(b) == 2:
                win+= circle((xb-0.5,yb-0.5), self.r, fill = True, color = Color('black'))
                
                #win+= circle((xb-0.5,yb-0.5), self.r, fill = False, color = Color('black'))
                #win+= text(lab,(xb-0.5,yb-0.525),color = Color('black'),fontsize = 18,rotation = 0)
            elif len(b) == 3:
                if b[2] == "p":
                    win+= circle((xb-0.5,yb-0.5), self.r, fill = False, color = Color('black'))
                    
                elif b[2] == "n":
                    win += line([((xb-0.5)-(self.r),(yb-0.5)+(self.r)),
                                     ((xb-0.5)+(self.r),(yb-0.5)+(self.r))],color=Color('black'))
                        
                    win += line([((xb-0.5)+(self.r),(yb-0.5)+(self.r)),
                                     ((xb-0.5)+(self.r),(yb-0.5)-(self.r))],color=Color('black'))
                        
                    win += line([((xb-0.5)+(self.r),(yb-0.5)-(self.r)),
                                     ((xb-0.5)-(self.r),(yb-0.5)-(self.r))],color=Color('black'))
                        
                    win += line([((xb-0.5)-(self.r),(yb-0.5)-(self.r)),
                                     ((xb-0.5)-(self.r),(yb-0.5)+(self.r))],color=Color('black'))
                                
        #Draw queues
        for k in range(len(self.queues)):
            queue = self.queues[k] 
            for j in range(1,len(queue)):
                N = len(self.queues)
                #maximum offset to avoid queues to be intersecting in the drawing
                par = (-1)/(5.0)
                #offset function
                off = (par)-(2*par/(1.0*N))*(k)
                off = 0.15
                #draw the line between elements of the queue
                win = self.draw_queue(win,queue[j-1],queue[j],self.case,off)

        win.axes(False)
        win.show()
        


    ## TASEP METHODS ##
    

    # returns the projection of the multiline queue onto a open-TASEP state 
    def projection(self):
        bot_row = [x for x in self.balls_coordinates if x[0] == 1]
        ans = [0 for i in range(self.size[0])]

        for b in bot_row:
            ind_b = find_queue(self,b)
            lab = len(self.queues[ind_b])
            if b[2] == "p":
                ans[b[1]-1] = int(lab/2.0)
            elif b[2] == "n":
                ans[b[1]-1] = (-1)*int(lab/2.0)

        return(ans)


    # computes the "epsilon_i" statistic on the multiline queue, i.e., the maximum number of nontrivial applications of "e_i"
    def epsilon(self,i):
        rw_rev = self.rw()

        aux_word = []

        for x in rw_rev:
            if x==i or x == (-1)*(i+1): 
                aux_word.append(1)
            elif x==i+1 or x==(-1)*i:
                aux_word.append(2)
            else:
                aux_word.append(0)

        pm_word = parenthesis_matching(aux_word)
        
        for y in range(len(pm_word)):
            if pm_word[y] != 0:
                aux_word[y] = 0
        
        return(len([x for x in aux_word if x==2]))

    
    # computes the "phi_i" statistic on the multiline queue, i.e., the maximum number of nontrivial applications of "f_i"
    def phi(self,i):
        rw_rev = self.rw()

        aux_word = []

        for x in rw_rev:
            if x==i or x == (-1)*(i+1): 
                aux_word.append(1)
            elif x==i+1 or x==(-1)*i:
                aux_word.append(2)
            else:
                aux_word.append(0)

        pm_word = parenthesis_matching(aux_word)
        
        for y in range(len(pm_word)):
            if pm_word[y] != 0:
                aux_word[y] = 0
        
        return(len([x for x in aux_word if x==1]))


    # computes the operator "f_i^j" on the multiline queue
    # returns the modified reading word of the multiline queue
    # 1 <= j <= self.phi(i)
    def ring(self,i,j):
        n = self.shape[0]
        # if i!=0 and i!=n:
        rw_rev = self.rw()
        sh = copy.deepcopy(self.shape)
        curr_word = copy.deepcopy(rw_rev)
        for k in range(j):
            curr_word = F_word(curr_word,i,sh,self.size[0])
        return(curr_word)

        
        
    ##  ALGEBRAIC METHODS  ##

    
    #returns the y-weight of the multiline queue. Does not need to be paired
    # Here we understand y[i] as x[i]^(1/2)
    def yweight(self,ys):
        wei = ys[0]**0
        for b in self.balls_coordinates:
            if b[2] == 'n':
                wei *= ys[b[1]-1]**(-1)
            else:
                wei *= ys[b[1]-1]
        return(wei)
    
    #returns the q-weight of the multiline queue. Requires pairing
    # Computes the major index of the multiline queue based on left bounces
    # case: see pairing cases
    def maj(self):
        ans = 0
        for queue in self.queues:
            for i in range(len(queue)-1):
                if queue[i][2] == 'p':
                    if queue[i+1][2] == 'n':
                        ans += int(len(queue)/2.0) - int(queue[i][0]/2.0)
                    elif queue[i+1][2] == 'p' and queue[i][1] < queue[i+1][1]:
                        ans += int(len(queue)/2.0) - int(queue[i][0]/2.0) + 1
                else:
                    if queue[i+1][2] == 'n' and queue[i][1] > queue[i+1][1]:
                        ans += int(len(queue)/2.0) - int(queue[i][0]/2.0)
        
        return(ans)

## Algebraic Combinatorics (in progress...)

In [2]:
def all_typeC_MLQs(lam,n):
    #Generate all products of KN columns to construct the MLQs
    list_crystals = []
    for i in range(len(lam)):
        list_crystals.append(crystals.KirillovReshetikhin(['C',n,1],lam[i],1))
    prod = crystals.TensorProduct(*list_crystals)

    #Compute the set of objects
    allMLQs = []
    for p in prod:
        bs = to_number_matrix(n,p)
        M = MultilineQueueC(bs)
        M.pair(0)
        allMLQs.append(M)

    return(allMLQs)

def all_nonleftbouncing_typeC_MLQs(lam,n):
    #Generate all products of KN columns to construct the MLQs
    list_crystals = []
    for i in range(len(lam)):
        list_crystals.append(crystals.KirillovReshetikhin(['C',n,1],lam[i],1))
    prod = crystals.TensorProduct(*list_crystals)

    #Compute the set of objects
    allMLQs = []
    for p in prod:
        bs = to_number_matrix(n,p)
        M = MultilineQueueC(bs)
        M.pair(0)
        if not M.is_left_bouncing():
            allMLQs.append(M)

    return(allMLQs)
    

In [2]:
def P_hat(n,lam):

    #Define the Laurent polynomial rings
    R_prime = LaurentPolynomialRing(coeffs_field,n,'y')
    R = LaurentPolynomialRing(coeffs_field,n,'x')
    xs = R.gens()
    ys = R_prime.gens()

    #Generate all products of KN columns to construct the MLQs
    list_crystals = []
    for i in range(len(lam)):
        list_crystals.append(crystals.KirillovReshetikhin(['C',n,1],lam[i],1))
    prod = crystals.TensorProduct(*list_crystals)

    #Compute the polynomial
    poly = 0
    for p in prod:
        balls = to_number_matrix(n,p)
        M = MultilineQueueC(balls)
        M.pair(0)
        
        y_exps = M.yweight(ys).exponents()[0]
        wx = xs[0]**0
        for i in range(n):
            wx *= xs[i]**(int(y_exps[i]/2))

        poly += wx

    return(poly)


def s_hat(n,lam):
    if lam == []:
        return(1)

    else:
        #Define the Laurent polynomial rings
        R_prime = LaurentPolynomialRing(coeffs_field,n,'y')
        R = LaurentPolynomialRing(coeffs_field,n,'x')
        xs = R.gens()
        ys = R_prime.gens()
            
        #Generate all products of KN columns to construct the MLQs
        list_crystals = []
        for i in range(len(lam)):
            list_crystals.append(crystals.KirillovReshetikhin(['C',n,1],lam[i],1))
        prod = crystals.TensorProduct(*list_crystals)
    
        #Compute the polynomial
        poly = 0
        for p in prod:
            balls = to_number_matrix(n,p)
            M = MultilineQueueC(balls)
            M.pair(0)
    
            if not M.is_left_bouncing():
                y_exps = M.yweight(ys).exponents()[0]
                wx = xs[0]**0
                for i in range(n):
                    wx *= xs[i]**(int(y_exps[i]/2))
                poly += wx
    
        return(poly)
        